# Customer Segments — типология клиентской базы

ДЗ6 (unsupervised): ~20 000 клиентов, один тариф `standard` (15 000 UZS/мес). Готового сегмента в базе нет — типы строим и защищаем сами.

Не используем: `seg_customers_360` (crm A/B/C/D), `seg_rfm_old`, `seg_transactions_archive`.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT = Path("..").resolve()
sys.path.insert(0, str(PROJECT / "scripts"))

from cluster import (
    CLUSTER_FEATURES,
    choose_k,
    k_metrics,
    load_active,
    make_matrix,
    run,
    scale,
)

OUT = PROJECT / "outputs"
DATA = PROJECT / "data"
plt.rcParams["figure.figsize"] = (8, 3.8)
plt.rcParams["axes.grid"] = False

print("project:", PROJECT.name)

project: customer_segments


## 1. Окно наблюдения

Данные за 2024–2025, клиенты пришли с 2019 по октябрь 2025. Среднее за 24 месяца смешает стаж и поведение: у части людей операций до 2024 в выгрузке просто нет.

| Параметр | Значение |
|----------|----------|
| Snapshot | 2025-11-30 (декабрь обрезан) |
| Lookback | 6 месяцев |
| Мин. стаж | 90 дней |
| Персональное окно | `max(window_start, signup_date)` |
| Интенсивность | в месяц окна, не сумма за 6 месяцев |

In [2]:
summary = pd.read_json(DATA / "features_summary.json", typ="series")
expl = pd.read_json(DATA / "exploration_results.json", typ="series")
print(summary.to_string())
print()
print("H1 vs H2 2025, корреляция оборота out:", expl["h1_h2_spend_corr"])
print("Слишком новые (signup после 2025-09-01):", expl["signup_after_cutoff"])
print("Нет ни одной операции за всё время:", expl["clients_no_txn"])

as_of                  2025-11-30 00:00:00.000000000
lookback_months        1970-01-01 00:00:06.000000000
min_tenure_days        1970-01-01 00:01:30.000000000
eligible               1970-01-01 05:28:47.000000000
active_in_window       1970-01-01 05:22:11.000000000
inactive_in_window     1970-01-01 00:06:36.000000000
short_history_active   1970-01-01 00:11:54.000000000
median_window_days     1970-01-01 00:03:05.000000000
median_spend_uzs       1970-11-10 11:00:33.000000000
p99_spend_uzs          1988-04-06 15:51:20.600001335

H1 vs H2 2025, корреляция оборота out: {'corr': 0.9851275862012441, 'n_both': 18782}
Слишком новые (signup после 2025-09-01): 273
Нет ни одной операции за всё время: 245


Корреляция оборота за первое и второе полугодие 2025 — **0.985** (18 782 человека в обоих). За год тип не меняется, шесть месяцев не склеивают двух разных людей.

Кого не кластеризуем: 273 слишком новых, 396 без операций в окне (из них 245 без операций вообще). Остаётся **19 331** активных.

## 2. Оси сегментации

Не все колонки подряд. Демография и стаж остались в профиле, но **не двигают** центроид — иначе получим «молодые / старые» или «давно с нами», а не типы пользования картой.

| Ось | Признаки | Зачем |
|-----|----------|--------|
| Интенсивность | `txn_per_month`, `spend_per_month` | как часто и на сколько *сейчас* |
| Каналы | `pos_share`, `ecom_share`, `atm_share` | где живёт карта; p2p не берём — доли суммируются в 1 |
| Деньги банка | `avg_balance`, `avg_monthly_revenue` | остаток и то, что банк уже зарабатывает |
| Глубина / свежесть | `product_cnt`, `recency_days` | продукты и пауза |

Выбросы (оборот до 7 млрд/мес): winsorize 1–99%, логарифм, `StandardScaler`. Направление в базе — `out`/`in`, не debit/credit.

In [3]:
df = load_active()
print("active:", len(df), "features:", CLUSTER_FEATURES)
print(df[CLUSTER_FEATURES].describe(percentiles=[0.5, 0.99]).T[["50%", "99%", "max"]].round(2))

active: 19331 features: ['txn_per_month', 'spend_per_month', 'pos_share', 'ecom_share', 'atm_share', 'avg_balance_uzs', 'avg_monthly_revenue_uzs', 'product_cnt', 'recency_days']
                                50%          99%           max
txn_per_month                 13.95       106.22  5.704900e+02
spend_per_month          4435248.97  94679926.36  7.358472e+09
pos_share                      0.46         0.94  1.000000e+00
ecom_share                     0.16         0.75  1.000000e+00
atm_share                      0.23         0.60  1.000000e+00
avg_balance_uzs          1621169.00  50421903.31  6.812813e+08
avg_monthly_revenue_uzs    40599.00    646243.04  5.331009e+07
product_cnt                    1.00         5.00  8.000000e+00
recency_days                   2.00        53.00  1.790000e+02


## 3. Типология и число классов

KMeans. Локоть — не аргумент: inertia всегда падает. Смотрим на три вещи:

1. Каждый класс ≥ 8% базы — иначе под него нельзя заводить тариф.
2. Типы должны требовать **разных** продуктов, не быть «чуть больше / чуть меньше».
3. Silhouette — поддержка, не судья.

In [4]:
x, _ = make_matrix(df)
z, _ = scale(x)
metrics = pd.read_csv(OUT / "k_metrics.csv")
print(metrics.to_string(index=False))
print("choose_k:", choose_k(metrics))

 k       inertia  silhouette  calinski_harabasz  davies_bouldin  min_share  max_share  tiny
 2 126371.041622    0.258379        7281.852526        1.484660   0.485231   0.514769     0
 3 107099.311902    0.246482        6034.834542        1.423647   0.098339   0.468367     0
 4  92218.730041    0.254823        5711.714196        1.459631   0.096788   0.455383     0
 5  82497.961985    0.264279        5357.596460        1.356441   0.094615   0.423879     0
 6  75563.940199    0.263628        5033.805875        1.347817   0.021106   0.408722     1
 7  70724.669447    0.212955        4702.021234        1.431991   0.021623   0.355491     1
 8  67137.752391    0.188130        4392.879921        1.476746   0.021623   0.272412     1
choose_k: 5


**K = 5.** Почему не соседи:

- **K = 2–3** — «активные / тихие». Для линейки тарифов бесполезно.
- **K = 4** — «хранят деньги» сливаются с «крупным оборотом» (ARI 5 vs 4 = 0.81). Это разные продукты: ставка на остаток vs премиум за оборот.
- **K = 6+** — появляется класс на 2% (~400 человек). Задание прямо говорит не делать тариф под 200 человек.

Silhouette 0.26 — типы пересекаются, это нормально для поведения карты. Доказательство качества — устойчивость и то, что квартили оборота этот рисунок не повторяют.

In [5]:
# Полный прогон пишет профили, ARI, экономику, модель.
summary = run(chosen_k=5)
prof = pd.read_csv(OUT / "cluster_profiles.csv")
print(prof[["segment", "n", "share", "txn_per_month", "spend_per_month",
            "pos_share", "ecom_share", "atm_share", "avg_balance_uzs"]].round(3).to_string(index=False))

K=5 names={4: 'Редкая карта', 0: 'Крупный оборот', 2: 'Онлайн', 3: 'Хранят деньги', 1: 'Повседневные'}
 k       inertia  silhouette  calinski_harabasz  davies_bouldin  min_share  max_share  tiny
 2 126371.041622    0.258379        7281.852526        1.484660   0.485231   0.514769     0
 3 107099.311902    0.246482        6034.834542        1.423647   0.098339   0.468367     0
 4  92218.730041    0.254823        5711.714196        1.459631   0.096788   0.455383     0
 5  82497.961985    0.264279        5357.596460        1.356441   0.094615   0.423879     0
 6  75563.940199    0.263628        5033.805875        1.347817   0.021106   0.408722     1
 7  70724.669447    0.212955        4702.021234        1.431991   0.021623   0.355491     1
 8  67137.752391    0.188130        4392.879921        1.476746   0.021623   0.272412     1
                                        check      ari     n
                другой random_state (7 vs 42) 0.999748 19331
 половина выборки (кластер vs исходные 

Пять типов активных + «Спящие» правилом:

| Тип | Одна фраза |
|-----|------------|
| Повседневные | Карта для снятия и магазинов, живут как все |
| Онлайн | Платят в интернете, почти не снимают |
| Крупный оборот | Мало людей — почти половина дохода банка |
| Хранят деньги | Редко платят, на карте лежит остаток |
| Редкая карта | Раз в месяц ткнули POS |
| Спящие | В окне тишина |

## 4. Сравнение с квартилями по обороту

Простое разбиение: Q1–Q4 по `spend_uzs`. Если наша типология — замаскированные квартили, ARI будет высоким, а внутри Q4 каналы не различаются.

In [6]:
print(pd.read_csv(OUT / "quartile_compare.csv").to_string(index=False))
print()
print("Доли квартилей внутри типа:")
print(pd.read_csv(OUT / "quartile_crosstab.csv").round(3).to_string(index=False))
print()
print("Только Q4 (крупный оборот) — каналы и остаток всё ещё разные:")
print(pd.read_csv(OUT / "quartile_q4_mix.csv").round(3).to_string(index=False))

                    metric    value
      silhouette_our_types 0.264279
silhouette_spend_quartiles 0.030520
    ARI_types_vs_quartiles 0.179414

Доли квартилей внутри типа:
 cluster    Q1    Q2    Q3    Q4
       0 0.000 0.002 0.006 0.992
       1 0.343 0.412 0.196 0.048
       2 0.023 0.193 0.442 0.341
       3 0.035 0.191 0.416 0.358
       4 0.990 0.010 0.000 0.001

Только Q4 (крупный оборот) — каналы и остаток всё ещё разные:
 cluster  atm_share  ecom_share  pos_share  avg_balance_uzs
       0      0.211       0.215      0.363      5512368.643
       1      0.313       0.102      0.431      1495741.429
       2      0.113       0.430      0.362      2058237.357
       3      0.163       0.198      0.531     12864104.857
       4      0.062       0.094      0.844      6125432.143


Silhouette квартилей на тех же осях — **0.03**, у типов — **0.26**. ARI между разбиениями **0.18**: это не одно и то же.

Киты почти все в Q4, редкая карта почти вся в Q1 — оборот *участвует*. Но повседневные / онлайн / хранят деньги живут в Q2–Q4 одновременно. Внутри верхнего квартиля у «хранят деньги» медианный остаток ~13 млн, у повседневных ~1.5 млн, у онлайн — 43% ecom. Квартиль этого не видит. **Типология лучше квартилей** — не красивее, а потому что меняет решение по тарифу.

## 5. Устойчивость (ARI)

In [7]:
stab = pd.read_csv(OUT / "stability_ari.csv")
print(stab.to_string(index=False))

                                        check      ari     n
                другой random_state (7 vs 42) 0.999748 19331
 половина выборки (кластер vs исходные метки) 0.979456  9665
                   без баланса и дохода банка 0.477839 19331
                без recency и числа продуктов 0.821315 19331
                            без долей каналов 0.533699 19331
окно без последнего месяца (as_of=2025-10-31) 0.874517 19072


| Проверка | ARI | Как читать |
|----------|----:|------------|
| Другой `random_state` | 1.00 | Центроиды не от зерна |
| Половина выборки | 0.98 | Типы не от «этих конкретных 19 тысяч» |
| Окно без ноября | 0.87 | Один месяц не переписывает тип |
| Без recency и продуктов | 0.82 | Лёгкий сдвиг набора — типы те же |
| Без каналов | 0.53 | Каналы — несущая ось, не украшение |
| Без баланса и дохода | 0.48 | Деньги банка тоже несущая ось |

Низкий ARI при выкидывании осей — это не провал. Это доказательство, что оси выбраны не «все подряд»: без них получается **другая** типология.

## 6. Правило для нового клиента

1. Собрать те же 9 признаков за его персональное окно (как в `sql/02_features.sql`).
2. Если 0 операций → тип **Спящие**.
3. Иначе: winsorize границами обучения → `log1p` → `StandardScaler` → ближайший центроид KMeans.

Пайплайн: `models/segment_pipeline.joblib`.

Проверка на 20% людей, которых не было в обучении центроидов:

In [8]:
import json
hold = json.loads((OUT / "holdout.json").read_text())
print(json.dumps(hold, ensure_ascii=False, indent=2))
print()
print((OUT / "assignment_rules.txt").read_text())

{
  "holdout_n": 3867,
  "ari_nearest_centroid_vs_train_labels": 0.9923932083861052,
  "ari_independent_kmeans_on_holdout": 0.9435565524039407,
  "tree_accuracy_holdout": 0.8763899663822085
}

Правило для нового клиента: те же 9 признаков за его персональное окно → winsorize 1–99% (границы с обучения) → log1p на тратах/операциях/остатке → StandardScaler → ближайший центроид KMeans.

Дерево ниже — шпаргалка для PM (глубина 3), не замена центроиду. Признаки в исходных единицах.

|--- ecom_share <= 0.252
|   |--- avg_monthly_revenue_uzs <= 68597.930
|   |   |--- txn_per_month <= 3.184
|   |   |   |--- class: 4
|   |   |--- txn_per_month >  3.184
|   |   |   |--- class: 1
|   |--- avg_monthly_revenue_uzs >  68597.930
|   |   |--- spend_per_month <= 21723749.000
|   |   |   |--- class: 3
|   |   |--- spend_per_month >  21723749.000
|   |   |   |--- class: 0
|--- ecom_share >  0.252
|   |--- txn_per_month <= 17.919
|   |   |--- avg_balance_uzs <= 3688435.875
|   |   |   |--- class: 4
|   |  

Ближайший центроид на holdout совпадает с «истинной» меткой обучения с ARI **0.99**. Независимо кластеризовать holdout — ARI **0.94**: типы существуют и на новых людях, это не запоминание обучающей выборки.

Дерево глубины 3 — шпаргалка для PM (точность 88%), не замена центроиду.

## 7. Экономика и предложение

`net_revenue` в витрине = interchange + float + комиссия перевода − ATM cost. Абонентка 15 000 **не входит** в эту сумму — её считаем отдельно.

In [9]:
eco = pd.read_csv(OUT / "economics.csv")
off = pd.read_csv(OUT / "tariff_offers.csv")
show = eco.merge(off[["segment", "tariff", "monthly_fee_uzs", "fee_delta_year_uzs", "hook"]], on="segment")
cols = ["segment", "n", "net_revenue_6m", "fee_revenue_6m", "tariff", "monthly_fee_uzs", "fee_delta_year_uzs"]
print(show[cols].to_string(index=False))
print()
print("Абонентка, дельта год, млн UZS:", round(show["fee_delta_year_uzs"].sum() / 1e6, 1))

       segment    n  net_revenue_6m  fee_revenue_6m   tariff  monthly_fee_uzs  fee_delta_year_uzs
  Повседневные 8194    1422812046.0       737460000 Стандарт            15000                   0
        Онлайн 4329    2264993270.0       389610000   Онлайн            19000           207792000
Крупный оборот 1854    6243914973.0       166860000  Премиум            30000           333720000
 Хранят деньги 3125    3182547696.0       281250000  Остаток            20000           187500000
  Редкая карта 1829      60018035.0       164610000     Лайт             5000          -219480000
        Спящие  396             0.0        35640000    Пауза                0           -71280000

Абонентка, дельта год, млн UZS: 438.3


| Тип | Людей | Доход банка 6 мес. | Тариф |
|-----|------:|-------------------:|-------|
| Повседневные | 8 194 | 1.42 млрд | Стандарт 15 000, кэшбэк POS |
| Онлайн | 4 329 | 2.26 млрд | Онлайн 19 000 + кэшбэк ecom |
| Крупный оборот | 1 854 | **6.24 млрд (47%)** | Премиум 30 000, не пугать дешёвым |
| Хранят деньги | 3 125 | 3.18 млрд (float) | Остаток 20 000 + ставка |
| Редкая карта | 1 829 | 60 млн | Лайт 5 000 |
| Спящие | 396 | 0 + абонентка | Пауза, 0 |

Дельта только по абонентке: **+438 млн UZS/год**. Это не главный P&L: киты уже приносят 350 тыс./мес. net без учёта платы. Главное — не отдать 47% дохода «наличному» тарифу и не держать 15 000 на карте, которой не пользуются.

273 новых в модель не входят: через 90 дней прогоняем то же правило.